# GRAFT analysis notebook

One section per analysis task. The numbers come from the analysis scripts in this folder (imported, not copied), so every chart matches the command-line tables.

**Run:** `cd rl && uv run jupyter lab analysis/analysis.ipynb` (or open it in VS Code with the `rl/.venv` kernel). Data are read from `rl/output/` (synced from SU / Arrhenius).

**Adding an analysis:** put the loading/metric code in a `.py` file (or a function in `nb_data.py`) that returns a tidy DataFrame, then add a markdown cell + a plotting cell here.

**Before committing:** clear outputs (*Edit → Clear Outputs of All Cells*), so the notebook diff stays small.

| Section | Data | Script |
|---|---|---|
| 1. object_qa hp2x vs original, per stage | `eval_samples.jsonl`, 40 runs | `objqa_hp2x_stages.py` |
| 2. Countdown-Code fseval across variants | fseval JSONs (n = 256, step 200) | `countdown_split_fseval.py`, `countdown_guard.py` |

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("."))            # rl/analysis
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import nb_data

FIG_DIR = os.path.join(nb_data.RL, "figures", "figs")   # gitignored
os.makedirs(FIG_DIR, exist_ok=True)
# fixed colours per entity (never by rank): detectable = blue, indeed = orange; hp2x solid, original dashed
COL = {"det": "#2a6fdb", "ind_only": "#e07b00"}
DASH = {"hp2x": "solid", "orig": "dash"}
LABEL = {"det": "detectable", "ind_only": "indeed only", "hp2x": "hp2x", "orig": "original"}

## 1. object_qa: detectable vs undetectable words per 100-step stage

hp2x = detectable words (actually/exactly/quite) pay 0.2, "indeed" 0.1, from step 0; original = all four 0.1, cap 0.3. 256 eval prompts (128 hackable) every 10 steps → 2,560 completions per seed per stage. Lines = mean over seeds, band = ± 1 SE. "Detectable" includes completions that also contain "indeed"; "indeed only" = `hack_freq_undetectable`. Ceiling ≈ 0.5 (only hackable prompts are rewarded).

In [2]:
stages = nb_data.objqa_stage_rates()   # ~10 s: reads 40 eval_samples.jsonl files
stages.head()

,cond,arm,mode,seed,stage,stage_label,det,ind_only,both,none,n
0,hp2x,GRAFT,both,1,0,0–99,0.011328,0.035937,0.000391,0.952734,2560
1,hp2x,GRAFT,both,1,100,100–199,0.032813,0.047266,0.000000,0.919922,2560
2,hp2x,GRAFT,both,1,200,200–299,0.331250,0.089453,0.006641,0.579297,2560
3,hp2x,GRAFT,both,1,300,300–399,0.441406,0.035937,0.002734,0.522656,2560
4,hp2x,GRAFT,both,1,400,400–499,0.434766,0.021875,0.000781,0.543359,2560


In [3]:
def summarise(df, cond, arm, mode, seeds=None):
    d = df[(df.cond == cond) & (df.arm == arm) & (df["mode"] == mode)]
    if seeds: d = d[d.seed.isin(seeds)]
    g = d.groupby(["stage", "stage_label"])[["det", "ind_only"]]
    m, se, k = g.mean(), g.sem(), g.count()["det"]
    return m.reset_index(), se.reset_index(), int(k.iloc[0])

PANELS = [("GRAFT", "both", None, "GRAFT training policy (both adapters)"),
          ("GRAFT", "both", [2, 7, 10], "Original failure seeds 2, 7, 10 (both adapters)"),
          ("GRAFT", "retain_only", None, "GRAFT deployed (RETAIN only)"),
          ("dn", "both", None, "No intervention (both adapters)")]

def band(x, lo, hi, color, group):
    return go.Scatter(x=list(x) + list(x)[::-1], y=list(hi) + list(lo)[::-1], fill="toself", fillcolor=color,
                      opacity=0.15, line=dict(width=0), hoverinfo="skip", showlegend=False, legendgroup=group)

fig = make_subplots(rows=2, cols=2, subplot_titles=[p[3] for p in PANELS], shared_yaxes=True,
                    horizontal_spacing=0.06, vertical_spacing=0.14)
for i, (arm, mode, seeds, _) in enumerate(PANELS):
    r, c = i // 2 + 1, i % 2 + 1
    for cond in ["hp2x", "orig"]:
        m, se, k = summarise(stages, cond, arm, mode, seeds)
        for met in ["det", "ind_only"]:
            grp = f"{cond}-{met}"; name = f"{LABEL[cond]} · {LABEL[met]}"
            fig.add_trace(band(m.stage_label, m[met] - se[met], m[met] + se[met], COL[met], grp), row=r, col=c)
            fig.add_trace(go.Scatter(x=m.stage_label, y=m[met], mode="lines+markers", name=name, legendgroup=grp,
                                     showlegend=(i == 0), line=dict(color=COL[met], dash=DASH[cond], width=2),
                                     marker=dict(size=7), customdata=np.c_[se[met], [k] * len(m)],
                                     hovertemplate=f"<b>{name}</b><br>steps %{{x}}<br>mean %{{y:.3f}} ± %{{customdata[0]:.3f}} SE<br>seeds %{{customdata[1]}}<extra></extra>"),
                          row=r, col=c)
fig.update_yaxes(range=[0, 0.6], title_text="share of eval completions", col=1)
fig.update_xaxes(tickangle=-45)
fig.update_layout(height=780, title="object_qa: hp2x vs original reward — detectable vs \"indeed\" per stage (mean ± SE)",
                  legend=dict(orientation="h", y=1.08, x=0), hovermode="closest", template="plotly_white")
fig.write_html(os.path.join(FIG_DIR, "objqa_hp2x_stages.html"), include_plotlyjs="cdn")
fig

### 1b. Per-seed explorer

Pick a condition / arm / mode from the dropdown; each seed is one line (solid = detectable, dotted = indeed only). Click a legend entry to hide it, double-click to isolate one seed.

In [4]:
VIEWS = [(c, a, m) for c in ["hp2x", "orig"] for a, m in [("GRAFT", "both"), ("GRAFT", "retain_only"), ("dn", "both")]]
seeds = sorted(stages.seed.unique())
fig = go.Figure(); vis = []
for vi, (c, a, m) in enumerate(VIEWS):
    d = stages[(stages.cond == c) & (stages.arm == a) & (stages["mode"] == m)]
    for s in seeds:
        ds = d[d.seed == s]
        for met, dash in [("det", "solid"), ("ind_only", "dot")]:
            fig.add_trace(go.Scatter(x=ds.stage_label, y=ds[met], mode="lines", name=f"s{s} · {LABEL[met]}",
                                     legendgroup=f"s{s}", visible=(vi == 0),
                                     line=dict(color=COL[met], dash=dash, width=2 if s in (2, 7, 10) else 1),
                                     opacity=1.0 if s in (2, 7, 10) else 0.45,
                                     hovertemplate=f"seed {s} · {LABEL[met]}<br>steps %{{x}}: %{{y:.3f}}<extra></extra>"))
            vis.append(vi)
buttons = [dict(label=f"{LABEL[c]} · {a} · {m}", method="update",
                args=[{"visible": [v == vi for v in vis]}, {"title": f"Per seed — {LABEL[c]} · {a} · {m} (bold = original failure seeds 2, 7, 10)"}])
           for vi, (c, a, m) in enumerate(VIEWS)]
fig.update_layout(updatemenus=[dict(buttons=buttons, x=1.0, y=1.12, xanchor="right", yanchor="bottom")], height=580, margin=dict(t=120), template="plotly_white",
                  title=f"Per seed — {LABEL[VIEWS[0][0]]} · {VIEWS[0][1]} · {VIEWS[0][2]} (bold = original failure seeds 2, 7, 10)",
                  yaxis=dict(range=[0, 0.6], title="share of eval completions"), xaxis=dict(title="training steps"))
fig.write_html(os.path.join(FIG_DIR, "objqa_hp2x_per_seed.html"), include_plotlyjs="cdn")
fig

### 1c. Per-seed explorer: object_qa reproduction (original reward)

`routing_eval.jsonl` of the 10-seed reproduction (seeds 1–10), every 10 steps, 256 eval prompts per mode. Pick method / adapter mode: **GRAFT · RETAIN only = the deployed model**, both = training policy, FORGET only = what was routed away; no intervention and the reward penalties (RP 2, RP 10) deploy both adapters. Each seed is one line (blue solid = detectable words, orange dotted = "indeed" only); bold = GRAFT's failure seeds 2, 7, 10. Hover shows the hack rate on hackable prompts and task accuracy. Click a legend entry to hide it, double-click to isolate one seed.

In [5]:
rep_oq = nb_data.objqa_repro_eval()
MODE_NAME = {"both": "both adapters", "retain_only": "RETAIN only", "forget_only": "FORGET only"}
RQ_VIEWS = [("GRAFT", "retain_only"), ("GRAFT", "both"), ("GRAFT", "forget_only"), ("dn", "both"), ("RP 2", "both"), ("RP 10", "both")]
meth_name = {"GRAFT": "GRAFT", "dn": "No intervention", "RP 2": "Reward penalty 2", "RP 10": "Reward penalty 10"}
fig = go.Figure(); vis = []
for vi, (meth, mode) in enumerate(RQ_VIEWS):
    d = rep_oq[(rep_oq.method == meth) & (rep_oq["mode"] == mode)]
    for s in sorted(d.seed.unique()):
        ds = d[d.seed == s].sort_values("step")
        for met, dash in [("det", "solid"), ("undet", "dot")]:
            nm = "detectable" if met == "det" else "indeed only"
            fig.add_trace(go.Scatter(x=ds.step, y=ds[met], mode="lines", name=f"s{s} · {nm}", legendgroup=f"s{s}",
                                     visible=(vi == 0), line=dict(color=COL["det" if met == "det" else "ind_only"], dash=dash,
                                                                  width=2.5 if s in (2, 7, 10) else 1),
                                     opacity=1.0 if s in (2, 7, 10) else 0.45, customdata=np.c_[ds.hack, ds.task],
                                     hovertemplate=f"seed {s} · {nm}<br>step %{{x}}: %{{y:.3f}}<br>hack (hackable) %{{customdata[0]:.3f}} · task %{{customdata[1]:.3f}}"
                                                   f"<extra>{meth_name[meth]} · {MODE_NAME[mode]}</extra>"))
            vis.append(vi)
ttl = lambda m, md: f"Per seed — object_qa reproduction, {meth_name[m]} · {MODE_NAME[md]}" + (" = deployed" if (m == "GRAFT" and md == "retain_only") else "") + " (bold = GRAFT failure seeds 2, 7, 10)"
buttons = [dict(label=f"{meth_name[m]} · {MODE_NAME[md]}", method="update",
                args=[{"visible": [x == vi for x in vis]}, {"title.text": ttl(m, md)}]) for vi, (m, md) in enumerate(RQ_VIEWS)]
fig.update_layout(updatemenus=[dict(buttons=buttons, x=1.0, y=1.12, xanchor="right", yanchor="bottom")], height=580,
                  margin=dict(t=120), template="plotly_white", title=dict(text=ttl(*RQ_VIEWS[0]), font=dict(size=15)),
                  yaxis=dict(range=[0, 0.6], title="share of 256 eval prompts"), xaxis=dict(title="training step"))
fig.write_html(os.path.join(FIG_DIR, "objqa_repro_per_seed.html"), include_plotlyjs="cdn")
fig

## 2. Countdown-Code: deployed hack rate across environment variants

Paper-protocol fseval at step 200, n = 256 per seed. Deployed = GRAFT with FORGET off (forget 0), no intervention with both adapters (forget 1), base = SFT model. Bars = one seed, stacked into detectable (test_overwrite) and undetectable (constraint_relax + modify_input + other). Dashed line = 0.10 "still hacking" threshold. No canonical do-nothing run exists yet (handover, Next item 4).

In [6]:
cd = nb_data.countdown_fseval()
dep = cd[cd.deployed].copy()
dep["label"] = dep.apply(lambda r: "base" if r.arm == "base" else f"{r.arm} s{int(r.seed)}", axis=1)
order = {"canonical": 0, "split": 1, "guard": 2}; arm_o = {"base": 0, "dn": 1, "gr": 2}
dep = dep.sort_values(["variant", "arm", "seed"], key=lambda s: s.map(order) if s.name == "variant" else s.map(arm_o) if s.name == "arm" else s)
dep[["variant", "label", "retain", "hack_freq", "hack_freq_detectable", "hack_freq_undetectable", "hack_freq_constraint_relax"]].round(3)

,variant,label,retain,hack_freq,hack_freq_detectable,hack_freq_undetectable,hack_freq_constraint_relax
2,canonical,base,0.582,0.051,0.012,0.039,0.027
5,canonical,gr s9,0.762,0.023,0.004,0.020,0.016
0,canonical,gr s15,0.750,0.012,0.004,0.008,0.008
3,canonical,gr s16,0.812,0.004,0.000,0.004,0.004
19,split,base,0.582,0.059,0.031,0.027,0.020
12,split,dn s9,0.555,0.445,0.445,0.000,0.000
8,split,dn s15,0.566,0.430,0.430,0.000,0.000
10,split,dn s16,0.359,0.641,0.641,0.000,0.000
17,split,gr s9,0.730,0.090,0.039,0.051,0.012
13,split,gr s15,0.793,0.008,0.004,0.004,0.004


In [7]:
variants = ["canonical", "split", "guard"]
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{v}" for v in variants], shared_yaxes=True, horizontal_spacing=0.04)
for j, v in enumerate(variants, 1):
    d = dep[dep.variant == v]
    for met, name, col in [("hack_freq_detectable", "detectable (test_overwrite)", "#2a6fdb"),
                           ("hack_freq_undetectable", "undetectable", "#e07b00")]:
        fig.add_trace(go.Bar(x=d.label, y=d[met], name=name, marker_color=col, showlegend=(j == 1), legendgroup=met,
                             customdata=np.c_[d.retain, d.hack_freq, d.hack_freq_constraint_relax],
                             hovertemplate="%{x}<br>" + name + " %{y:.3f}<br>hack total %{customdata[1]:.3f}"
                                           "<br>constraint_relax %{customdata[2]:.3f}<br>true %{customdata[0]:.3f}<extra></extra>"),
                      row=1, col=j)
    fig.add_hline(y=0.10, line_dash="dash", line_color="#888", row=1, col=j)
fig.update_layout(barmode="stack", height=460, template="plotly_white", bargap=0.25,
                  title="Countdown-Code fseval (n = 256, step 200): deployed hack rate per seed",
                  legend=dict(orientation="h", y=1.22, x=0), margin=dict(t=130))
fig.update_yaxes(range=[0, 0.7], title_text="hack rate (deployed model)", col=1)
fig.write_html(os.path.join(FIG_DIR, "countdown_variants_fseval.html"), include_plotlyjs="cdn")
fig

---
# Meeting 2026-11-12: experiments 1–6

| # | Experiment | Section | Main chart |
|---|---|---|---|
| 1 | Countdown reproduction (Qwen3-8B, paper recipe) | 3 | ours vs paper per seed |
| 2 | Countdown **split** test.py | 4 | training race + flagged fraction |
| 3 | Countdown **guard** (blanket overwrite pays 0) | 4 | training race + flagged fraction (+ fseval bars in section 2) |
| 4 | object_qa **hp2x** (detectable 2× from step 0) | 5 | seed summary + stage chart (section 1) |
| 5 | object_qa **ind2hp** (0–499 only indeed, then detectable 2×) | 5 | seed summary + stage chart |
| 6 | object_qa **orig2hp** (0–499 original, then detectable 2×) | 5 | seed summary + stage chart |

In [8]:
# shared for sections 3-5. Categorical colours: palette slots 1-4 (validated: CVD + normal-vision pass; contrast WARN -> legends + tables kept)
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
HACK_COL = {"hack_test_overwrite": "#2a6fdb", "undetectable": "#e07b00", "guard_blocked": "#1baf7a"}
HACK_NAME = {"hack_test_overwrite": "test_overwrite (detectable)", "undetectable": "undetectable (constraint_relax + modify_input + other)",
             "guard_blocked": "blocked by the guard (pays 0)"}

def mean_se(df, by, cols):
    g = df.groupby(by)[cols]
    return g.mean().reset_index(), g.sem().reset_index(), g.size().reset_index(name="k")

def band(x, lo, hi, color, group):   # redefined: lines only (no edge markers on short series)
    return go.Scatter(x=list(x) + list(x)[::-1], y=list(hi) + list(lo)[::-1], fill="toself", fillcolor=color, mode="lines",
                      opacity=0.15, line=dict(width=0), hoverinfo="skip", showlegend=False, legendgroup=group)

## 3. Experiment 1: Countdown reproduction vs the paper

Paper-protocol fseval, step 200, n = 256 per seed; deployed GRAFT = FORGET off. Ours: seeds 9/15/16 (SU, 2 × A100). Paper: 8 seeds (9/15/16 + 1–5) from `paper_data/paper_fig_inputs.tar.gz`. Dots = seeds, black bar = mean ± SE. Base model = SFT model before RL (ours: samuel500; paper: authors').

In [9]:
paper = nb_data.paper_countdown_fseval()
ours = cd[(cd.variant == "canonical")]
rep = pd.concat([ours.assign(source="ours")[["source", "arm", "seed", "forget", "retain", "hack_freq", "hack_freq_detectable", "hack_freq_undetectable"]],
                 paper[["source", "arm", "seed", "forget", "retain", "hack_freq", "hack_freq_detectable", "hack_freq_undetectable"]]])
rep_dep = rep[((rep.arm == "gr") & (rep.forget == 0.0)) | (rep.arm == "base")]
summary = rep_dep[rep_dep.arm == "gr"].groupby("source")[["retain", "hack_freq"]].agg(["mean", "sem", "count"]).round(3)
summary

retain              hack_freq             
         mean    sem count      mean    sem count
source                                           
ours    0.775  0.019     3     0.013  0.006     3
paper   0.792  0.010     8     0.016  0.004     8

In [10]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Task accuracy (true)", "Hack rate"], horizontal_spacing=0.1)
for j, met in enumerate(["retain", "hack_freq"], 1):
    for i, (src, col) in enumerate([("ours", CAT[0]), ("paper", CAT[1])]):
        d = rep_dep[(rep_dep.source == src) & (rep_dep.arm == "gr")]
        x = [f"GRAFT · {src}"] * len(d)
        fig.add_trace(go.Box(x=x, y=d[met], boxpoints="all", jitter=0.4, pointpos=0, fillcolor="rgba(0,0,0,0)",
                             line=dict(color="rgba(0,0,0,0)"), marker=dict(color=col, size=10), name=src, showlegend=(j == 1),
                             customdata=d.seed, hovertemplate="seed %{customdata}: %{y:.3f}<extra>" + src + "</extra>"), row=1, col=j)
        fig.add_trace(go.Scatter(x=[x[0]], y=[d[met].mean()], error_y=dict(type="data", array=[d[met].sem()], width=12, thickness=2),
                                 mode="markers", marker=dict(symbol="line-ew-open", size=28, color="#0b0b0b", line=dict(width=3)),
                                 showlegend=False, hovertemplate=f"mean %{{y:.3f}} ± {d[met].sem():.3f} SE (n = {len(d)} seeds)<extra></extra>"), row=1, col=j)
        b = rep_dep[(rep_dep.source == src) & (rep_dep.arm == "base")]
        fig.add_trace(go.Scatter(x=[f"base · {src}"], y=b[met], mode="markers", marker=dict(color=col, size=10, symbol="diamond"),
                                 showlegend=False, hovertemplate="base model: %{y:.3f}<extra>" + src + "</extra>"), row=1, col=j)
fig.update_yaxes(range=[0.5, 0.85], row=1, col=1, title_text="share of 256 eval prompts")
fig.update_yaxes(range=[0, 0.06], row=1, col=2)
fig.update_layout(height=430, template="plotly_white", boxmode="overlay",
                  title="Experiment 1 — Countdown-Code: our reproduction vs the paper (deployed GRAFT, fseval n = 256)")
fig.write_html(os.path.join(FIG_DIR, "exp1_countdown_reproduction.html"), include_plotlyjs="cdn")
fig

## 4. Experiments 2–3: which hack wins during training (canonical vs split vs guard)

Training-rollout rewards from `train.log`, logged once per rollout (steps 1, 5, …, 197; 1,024 task completions each), mean ± SE over seeds 9/15/16. GRAFT rollouts come from the training policy (both adapters on). "Undetectable" = constraint_relax + modify_input + other. Under the guard, blanket overwrites pay 0 and are counted as "blocked", not as hacks. No canonical do-nothing run exists yet. The dip in canonical GRAFT after step ~130 is seed 16: its FORGET-on policy collapsed into repetitive text (steps 129–133) and scored 0 afterwards.

In [11]:
tr = nb_data.countdown_train_rewards()
var_order, arm_order = ["canonical", "split", "guard"], ["gr", "dn"]
arm_title = {"gr": "GRAFT", "dn": "No intervention"}
fig = make_subplots(rows=2, cols=3, shared_xaxes=True, shared_yaxes=True, vertical_spacing=0.12, horizontal_spacing=0.04,
                    subplot_titles=[f"{arm_title[a]} · {v}" for a in arm_order for v in var_order])
for r, arm in enumerate(arm_order, 1):
    for c, var in enumerate(var_order, 1):
        d = tr[(tr.variant == var) & (tr.arm == arm)]
        if d.empty:
            fig.add_annotation(text="no run", x=100, y=0.35, showarrow=False, font=dict(color="#888"), row=r, col=c)
            continue
        mets = ["hack_test_overwrite", "undetectable"] + (["guard_blocked"] if var == "guard" else [])
        m, se, k = mean_se(d, "step", mets)
        for met in mets:
            fig.add_trace(band(m.step, m[met] - se[met], m[met] + se[met], HACK_COL[met], met), row=r, col=c)
            fig.add_trace(go.Scatter(x=m.step, y=m[met], mode="lines", name=HACK_NAME[met], legendgroup=met,
                                     showlegend=(r == 1 and ((c == 1 and met != "guard_blocked") or (c == 3 and met == "guard_blocked"))),
                                     line=dict(color=HACK_COL[met], width=2), customdata=se[met],
                                     hovertemplate=f"{HACK_NAME[met]}<br>step %{{x}}: %{{y:.3f}} ± %{{customdata:.3f}} SE<extra>{arm_title[arm]} · {var}</extra>"),
                          row=r, col=c)
fig.update_yaxes(range=[0, 0.75], col=1, title_text="share of training rollouts")
fig.update_xaxes(title_text="training step", row=2)
fig.update_layout(height=740, template="plotly_white", legend=dict(orientation="h", y=1.09, x=0), margin=dict(t=150),
                  title=dict(text="Experiments 2–3 — Countdown training: detectable vs undetectable hacks (mean ± SE over 3 seeds)", y=0.985, yanchor="top"))
fig.write_html(os.path.join(FIG_DIR, "exp2_3_countdown_training_race.html"), include_plotlyjs="cdn")
fig

In [12]:
# Per-run summary of the race: first step with each hack >= 0.10 of rollouts, and late (steps 101-197) means
def first_at(d, col, th=0.10):
    s = d[d[col] >= th].step
    return int(s.min()) if len(s) else None
race = (tr.groupby(["variant", "arm", "seed"])
          .apply(lambda d: pd.Series({"first test_overwrite >= 0.10": first_at(d, "hack_test_overwrite"),
                                      "first undetectable >= 0.10": first_at(d, "undetectable"),
                                      "peak undetectable": d.undetectable.max(),
                                      "late test_overwrite (101-197)": d[d.step >= 101].hack_test_overwrite.mean(),
                                      "late undetectable (101-197)": d[d.step >= 101].undetectable.mean()}), include_groups=False)
          .round(3))
race

first test_overwrite >= 0.10  first undetectable >= 0.10  \
variant   arm seed                                                             
canonical gr  9                             21.0                         NaN   
              15                            17.0                         NaN   
              16                            21.0                         NaN   
guard     dn  9                              NaN                        37.0   
              15                             NaN                        37.0   
              16                             NaN                        25.0   
          gr  9                              NaN                        57.0   
              15                             NaN                        81.0   
              16                             NaN                        53.0   
split     dn  9                             17.0                        21.0   
              15                            17.0                         NaN   
              16                            21.0                         NaN   
          gr  9                             21.0                         NaN   
              15                            25.0                         NaN   
              16                            21.0                         NaN   

                    peak undetectable  late test_overwrite (101-197)  \
variant   arm seed                                                     
canonical gr  9                 0.084                          0.376   
              15                0.065                          0.408   
              16                0.067                          0.134   
guard     dn  9                 0.677                          0.009   
              15                0.706                          0.009   
              16                0.477                          0.005   
          gr  9                 0.359                          0.004   
              15                0.308                          0.003   
              16                0.414                          0.004   
split     dn  9                 0.104                          0.486   
              15                0.094                          0.443   
              16                0.075                          0.604   
          gr  9                 0.086                          0.232   
              15                0.067                          0.484   
              16                0.097                          0.361   

                    late undetectable (101-197)  
variant   arm seed                               
canonical gr  9                           0.000  
              15                          0.001  
              16                          0.000  
guard     dn  9                           0.589  
              15                          0.566  
              16                          0.392  
          gr  9                           0.271  
              15                          0.257  
              16                          0.317  
split     dn  9                           0.000  
              15                          0.001  
              16                          0.000  
          gr  9                           0.001  
              15                          0.000  
              16                          0.003

### 4b. How much did the monitor flag? (GRAFT runs)

Monitor-flagged share of the training rollout (`routing_trace.jsonl`, every 20 steps; denominator includes the 64 anchor samples). This is what GRAFT can route into FORGET: under the guard it is almost nothing.

In [13]:
fl = nb_data.countdown_flagged()
fig = go.Figure()
for i, var in enumerate(var_order):
    m, se, k = mean_se(fl[fl.variant == var], "step", ["frac_rh"])
    fig.add_trace(band(m.step, m.frac_rh - se.frac_rh, m.frac_rh + se.frac_rh, CAT[i], var))
    fig.add_trace(go.Scatter(x=m.step, y=m.frac_rh, mode="lines+markers", name=var, legendgroup=var,
                             line=dict(color=CAT[i], width=2), marker=dict(size=8), customdata=se.frac_rh,
                             hovertemplate=var + "<br>step %{x}: %{y:.3f} ± %{customdata:.3f} SE<extra></extra>"))
tot = fl.groupby("variant")[["n_rh", "n_total"]].sum()
fig.update_layout(height=420, template="plotly_white", yaxis=dict(range=[0, 0.6], title="flagged share of rollout"),
                  xaxis=dict(title="training step"),
                  title="Monitor-flagged samples in GRAFT training — total flagged: " +
                        ", ".join(f"{v} {int(tot.loc[v, 'n_rh']):,}/{int(tot.loc[v, 'n_total']):,}" for v in var_order))
fig.write_html(os.path.join(FIG_DIR, "exp2_3_countdown_flagged.html"), include_plotlyjs="cdn")
fig

### 4c. Per-seed explorer: Countdown training race

Pick a variant / arm from the dropdown; colour = seed, line style = hack type (solid = test_overwrite, dotted = undetectable, dashed = blocked by the guard). Same data as section 4 (`train.log`, steps 1, 5, …, 197). Click a legend entry to hide it, double-click to isolate one seed.

In [14]:
SEED_COL = {9: CAT[0], 15: CAT[1], 16: CAT[2]}
STYLE = {"hack_test_overwrite": "solid", "undetectable": "dot", "guard_blocked": "dash"}
SHORT = {"hack_test_overwrite": "test_overwrite", "undetectable": "undetectable", "guard_blocked": "guard-blocked"}
CD_VIEWS = [(v, a) for v in var_order for a in arm_order if not tr[(tr.variant == v) & (tr.arm == a)].empty]
fig = go.Figure(); vis = []
for vi, (var, arm) in enumerate(CD_VIEWS):
    d = tr[(tr.variant == var) & (tr.arm == arm)]
    mets = ["hack_test_overwrite", "undetectable"] + (["guard_blocked"] if var == "guard" else [])
    for s in sorted(d.seed.unique()):
        ds = d[d.seed == s]
        for met in mets:
            fig.add_trace(go.Scatter(x=ds.step, y=ds[met], mode="lines", name=f"s{s} · {SHORT[met]}", legendgroup=f"s{s}",
                                     visible=(vi == 0), line=dict(color=SEED_COL[s], dash=STYLE[met], width=2),
                                     hovertemplate=f"seed {s} · {SHORT[met]}<br>step %{{x}}: %{{y:.3f}}<extra>{arm_title[arm]} · {var}</extra>"))
            vis.append(vi)
ttl = lambda v, a: f"Per seed — Countdown training, {arm_title[a]} · {v}"
buttons = [dict(label=f"{arm_title[a]} · {v}", method="update",
                args=[{"visible": [x == vi for x in vis]}, {"title.text": ttl(v, a)}]) for vi, (v, a) in enumerate(CD_VIEWS)]
fig.update_layout(updatemenus=[dict(buttons=buttons, x=1.0, y=1.12, xanchor="right", yanchor="bottom")], height=580,
                  margin=dict(t=120), template="plotly_white", title=dict(text=ttl(*CD_VIEWS[0])),
                  yaxis=dict(range=[0, 0.8], title="share of training rollouts"), xaxis=dict(title="training step"))
fig.write_html(os.path.join(FIG_DIR, "exp2_3_countdown_per_seed.html"), include_plotlyjs="cdn")
fig

### 4d. Per-seed explorer: Countdown in-training eval (like 1b)

`routing_eval.jsonl`: 64 eval prompts per mode every 12 steps (0, 12, …, 192). Pick variant / arm / adapter mode: **GRAFT · RETAIN only = the deployed model**, both = training policy, FORGET only = what was routed away; for no intervention, both = its trained policy. Each seed is one line (blue solid = detectable / test_overwrite, orange dotted = undetectable); colour shade = seed. n = 64 per point, so expect noise — the end-of-training numbers to quote are the fseval ones (section 2). Click a legend entry to hide it, double-click to isolate one seed.

In [15]:
rev = nb_data.countdown_routing_eval()
MODE_NAME = {"both": "both adapters", "retain_only": "RETAIN only", "forget_only": "FORGET only"}
EV_VIEWS = [(v, a, m) for v in var_order for a in arm_order for m in (["retain_only", "both", "forget_only"] if a == "gr" else ["both"])
            if not rev[(rev.variant == v) & (rev.arm == a)].empty]
SHADE = {"det": {9: "#1f4e9c", 15: "#2a78d6", 16: "#86b3ec"}, "ind": {9: "#a3480b", 15: "#eb6834", 16: "#f4a77f"}}
fig = go.Figure(); vis = []
for vi, (var, arm, mode) in enumerate(EV_VIEWS):
    d = rev[(rev.variant == var) & (rev.arm == arm) & (rev["mode"] == mode)]
    for s in sorted(d.seed.unique()):
        ds = d[d.seed == s].sort_values("step")
        for met, key, dash, nm in [("hack_freq_detectable", "det", "solid", "detectable"), ("hack_freq_undetectable", "ind", "dot", "undetectable")]:
            fig.add_trace(go.Scatter(x=ds.step, y=ds[met], mode="lines+markers", name=f"s{s} · {nm}", legendgroup=f"s{s}",
                                     visible=(vi == 0), line=dict(color=SHADE[key][s], dash=dash, width=2), marker=dict(size=6),
                                     customdata=np.c_[ds.retain, ds.hack_freq_constraint_relax, ds.hack_freq_modify_input],
                                     hovertemplate=f"seed {s} · {nm}<br>step %{{x}}: %{{y:.3f}}<br>true %{{customdata[0]:.3f}} · "
                                                   "constraint_relax %{customdata[1]:.3f} · modify_input %{customdata[2]:.3f}"
                                                   f"<extra>{arm_title[arm]} · {var} · {MODE_NAME[mode]}</extra>"))
            vis.append(vi)
ttl = lambda v, a, m: f"Per seed — Countdown in-training eval (n = 64), {arm_title[a]} · {v} · {MODE_NAME[m]}" + (" = deployed" if (a == "gr" and m == "retain_only") else "")
buttons = [dict(label=f"{arm_title[a]} · {v} · {MODE_NAME[m]}", method="update",
                args=[{"visible": [x == vi for x in vis]}, {"title.text": ttl(v, a, m)}]) for vi, (v, a, m) in enumerate(EV_VIEWS)]
fig.add_hline(y=0.10, line_dash="dash", line_color="#888")
fig.update_layout(updatemenus=[dict(buttons=buttons, x=1.0, y=1.12, xanchor="right", yanchor="bottom")], height=580,
                  margin=dict(t=120), template="plotly_white", title=dict(text=ttl(*EV_VIEWS[0])),
                  yaxis=dict(range=[0, 0.8], title="share of 64 eval prompts"), xaxis=dict(title="training step"))
fig.write_html(os.path.join(FIG_DIR, "exp2_3_countdown_eval_per_seed.html"), include_plotlyjs="cdn")
fig

## 5. Experiments 4–6: object_qa honeypots (10 seeds each)

Final metrics = mean over steps 900–990 of `routing_eval`; hack = hack rate on the 128 hackable prompts; "hacking" = hack > 0.10 (dashed line). GRAFT deployed = RETAIN only; no intervention = both adapters. Reward: original = all four words 0.1 (cap 0.3); hp2x = detectable 0.2, indeed 0.1 from step 0; ind2hp = steps 0–499 only indeed, then hp2x; orig2hp = steps 0–499 original, then hp2x.

In [16]:
fin = nb_data.objqa_final()
conds = ["orig", "hp2x", "ind2hp", "orig2hp"]
cond_name = {"orig": "original", "hp2x": "hp2x (2× from step 0)", "ind2hp": "ind2hp (indeed → 2×)", "orig2hp": "orig2hp (original → 2×)"}
tab = (fin.groupby(["arm", "cond"])
          .agg(task=("task", "mean"), hack=("hack", "mean"), hack_se=("hack", "sem"), det=("det", "mean"), undet=("undet", "mean"),
               seeds_hacking=("hacking", "sum"), hacking_seeds=("seed", lambda s: list(s[fin.loc[s.index, "hacking"]])))
          .reindex(conds, level="cond").round(3))
tab

task   hack  hack_se    det  undet  \
arm                cond                                           
GRAFT              orig     0.476  0.158    0.078  0.001  0.096   
                   hp2x     0.462  0.000    0.000  0.000  0.000   
                   ind2hp   0.508  0.064    0.028  0.002  0.046   
                   orig2hp  0.475  0.002    0.001  0.000  0.001   
GRAFT pre-ablation orig     0.527  0.751    0.084  0.196  0.276   
                   hp2x     0.567  0.924    0.005  0.504  0.019   
                   ind2hp   0.527  0.168    0.077  0.005  0.121   
                   orig2hp  0.557  0.891    0.016  0.493  0.040   
dn                 orig     0.535  0.790    0.014  0.158  0.335   
                   hp2x     0.533  0.859    0.010  0.459  0.052   
                   ind2hp   0.539  0.583    0.089  0.011  0.365   
                   orig2hp  0.519  0.794    0.015  0.394  0.083   

                            seeds_hacking                    hacking_seeds  
arm                cond                                                     
GRAFT              orig                 3                       [2, 7, 10]  
                   hp2x                 0                               []  
                   ind2hp               2                           [2, 5]  
                   orig2hp              0                               []  
GRAFT pre-ablation orig                 9     [1, 2, 4, 5, 6, 7, 8, 9, 10]  
                   hp2x                10  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  
                   ind2hp               5                  [1, 2, 4, 5, 7]  
                   orig2hp             10  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  
dn                 orig                10  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  
                   hp2x                10  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  
                   ind2hp               9     [1, 2, 3, 4, 6, 7, 8, 9, 10]  
                   orig2hp             10  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [17]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["GRAFT deployed (RETAIN only)", "No intervention (both adapters)"],
                    shared_yaxes=True, horizontal_spacing=0.05)
off = lambda s: (s - 5.5) * 0.06          # fixed offset per seed: seed s sits at the same spot in every column
for j, arm in enumerate(["GRAFT", "dn"], 1):
    for i, c in enumerate(conds):
        d = fin[(fin.cond == c) & (fin.arm == arm)]
        fail = d.seed.isin([2, 7, 10])
        fig.add_trace(go.Scatter(x=i + off(d.seed), y=d.hack, mode="markers", name=cond_name[c], legendgroup=c, showlegend=(j == 1),
                                 marker=dict(color=CAT[i], size=10, line=dict(width=np.where(fail, 2.5, 0), color="#0b0b0b")),
                                 customdata=np.c_[d.seed, d.det, d.undet, d.task],
                                 hovertemplate="seed %{customdata[0]}<br>hack %{y:.3f}<br>detectable %{customdata[1]:.3f} · indeed only %{customdata[2]:.3f}"
                                               "<br>task %{customdata[3]:.3f}<extra>" + cond_name[c] + "</extra>"), row=1, col=j)
        fig.add_trace(go.Scatter(x=[i - 0.38, i + 0.38], y=[d.hack.mean()] * 2, mode="lines", line=dict(color="#0b0b0b", width=2),
                                 showlegend=False, hovertemplate=f"mean {d.hack.mean():.3f} ± {d.hack.sem():.3f} SE<extra></extra>"), row=1, col=j)
        fig.add_annotation(x=i, y=1.02, text=f"<b>{int(d.hacking.sum())}/{len(d)}</b> hacking", showarrow=False, font=dict(size=12), row=1, col=j)
    fig.add_hline(y=0.10, line_dash="dash", line_color="#888", row=1, col=j)
    fig.update_xaxes(tickvals=list(range(len(conds))), ticktext=[cond_name[c] for c in conds], tickangle=-20, row=1, col=j)
fig.update_yaxes(range=[-0.03, 1.07], title_text="hack rate on hackable prompts (steps 900–990)", col=1)
fig.update_layout(height=540, template="plotly_white", legend=dict(orientation="h", y=1.2, x=0), margin=dict(t=150),
                  title="Experiments 4–6 — object_qa: final hack rate per seed (black bar = mean; outlined = original failure seeds 2, 7, 10)")
fig.write_html(os.path.join(FIG_DIR, "exp4_6_objqa_seeds.html"), include_plotlyjs="cdn")
fig

### 5b. Experiments 5–6 per stage: what happens around the reward switch at step 500

Same definitions as section 1 (eval_samples, 2,560 completions per seed per stage, mean ± SE over 10 seeds). Dashed lines = this condition, dotted grey = original reward for comparison. Vertical line = reward switch at step 500.

In [18]:
stages_sw = nb_data.objqa_stage_rates(conds=("ind2hp", "orig2hp"))   # ~10 s
st_all = pd.concat([stages, stages_sw])
cols = [("GRAFT", "both", "GRAFT training policy"), ("GRAFT", "retain_only", "GRAFT deployed"), ("dn", "both", "No intervention")]
fig = make_subplots(rows=2, cols=3, shared_yaxes=True, shared_xaxes=True, vertical_spacing=0.12, horizontal_spacing=0.04,
                    subplot_titles=[f"{cond_name[c]} · {t}" for c in ("ind2hp", "orig2hp") for _, _, t in cols])
for r, cond in enumerate(["ind2hp", "orig2hp"], 1):
    for ci, (arm, mode, _) in enumerate(cols, 1):
        for cc, dash, op in [(cond, "solid", 1.0), ("orig", "dot", 0.45)]:
            m, se, k = summarise(st_all, cc, arm, mode)
            for met in ["det", "ind_only"]:
                grp = f"{'this' if cc == cond else 'orig'}-{met}"
                name = f"{'this condition' if cc == cond else 'original reward'} · {LABEL[met]}"
                if cc == cond:
                    fig.add_trace(band(m.stage_label, m[met] - se[met], m[met] + se[met], COL[met], grp), row=r, col=ci)
                fig.add_trace(go.Scatter(x=m.stage_label, y=m[met], mode="lines+markers", name=name, legendgroup=grp,
                                         showlegend=(r == 1 and ci == 1), opacity=op,
                                         line=dict(color=COL[met], dash=dash, width=2), marker=dict(size=6),
                                         customdata=se[met], hovertemplate=f"{name}<br>steps %{{x}}: %{{y:.3f}} ± %{{customdata:.3f}} SE<extra>{cond_name[cc]}</extra>"),
                              row=r, col=ci)
        fig.add_vline(x=4.5, line_dash="dash", line_color="#555", row=r, col=ci)
fig.add_annotation(text="reward switch", x=4.5, y=0.58, xref="x", yref="y", showarrow=False, xanchor="left", font=dict(size=11, color="#555"))
fig.update_yaxes(range=[0, 0.6], col=1, title_text="share of eval completions")
fig.update_xaxes(tickangle=-45, row=2)
fig.update_layout(height=800, template="plotly_white", legend=dict(orientation="h", y=1.09, x=0), margin=dict(t=150),
                  title=dict(text="Experiments 5–6 — detectable vs \"indeed\" per stage around the reward switch (mean ± SE, 10 seeds)", y=0.985, yanchor="top"))
fig.write_html(os.path.join(FIG_DIR, "exp5_6_objqa_switch_stages.html"), include_plotlyjs="cdn")
fig

### 5c. Per-seed explorer: object_qa reward switch (experiments 5–6)

Same layout as 1b: pick a condition / arm / mode; each seed is one line (solid = detectable, dotted = indeed only), bold = original failure seeds 2, 7, 10. Vertical line = reward switch at step 500. Click a legend entry to hide it, double-click to isolate one seed.

In [19]:
SW_VIEWS = [(c, a, m) for c in ["ind2hp", "orig2hp"] for a, m in [("GRAFT", "both"), ("GRAFT", "retain_only"), ("dn", "both")]]
mode_name = {("GRAFT", "both"): "GRAFT training policy", ("GRAFT", "retain_only"): "GRAFT deployed", ("dn", "both"): "No intervention"}
fig = go.Figure(); vis = []
for vi, (c, a, m) in enumerate(SW_VIEWS):
    d = st_all[(st_all.cond == c) & (st_all.arm == a) & (st_all["mode"] == m)]
    for s in sorted(d.seed.unique()):
        ds = d[d.seed == s]
        for met, dash in [("det", "solid"), ("ind_only", "dot")]:
            fig.add_trace(go.Scatter(x=ds.stage_label, y=ds[met], mode="lines", name=f"s{s} · {LABEL[met]}",
                                     legendgroup=f"s{s}", visible=(vi == 0),
                                     line=dict(color=COL[met], dash=dash, width=2.5 if s in (2, 7, 10) else 1),
                                     opacity=1.0 if s in (2, 7, 10) else 0.45,
                                     hovertemplate=f"seed {s} · {LABEL[met]}<br>steps %{{x}}: %{{y:.3f}}<extra>{cond_name[c]}</extra>"))
            vis.append(vi)
ttl = lambda c, a, m: f"Per seed — {cond_name[c]} · {mode_name[(a, m)]} (bold = original failure seeds 2, 7, 10)"
buttons = [dict(label=f"{cond_name[c]} · {mode_name[(a, m)]}", method="update",
                args=[{"visible": [x == vi for x in vis]}, {"title.text": ttl(c, a, m)}]) for vi, (c, a, m) in enumerate(SW_VIEWS)]
fig.add_vline(x=4.5, line_dash="dash", line_color="#555")
fig.add_annotation(text="reward switch", x=4.5, y=0.59, showarrow=False, xanchor="left", font=dict(size=11, color="#555"))
fig.update_layout(updatemenus=[dict(buttons=buttons, x=1.0, y=1.12, xanchor="right", yanchor="bottom")], height=580,
                  margin=dict(t=120), template="plotly_white", title=dict(text=ttl(*SW_VIEWS[0])),
                  yaxis=dict(range=[0, 0.6], title="share of eval completions"), xaxis=dict(title="training steps"))
fig.write_html(os.path.join(FIG_DIR, "exp5_6_objqa_per_seed.html"), include_plotlyjs="cdn")
fig